# Processamento e Análise Estatística — Loteca

Segunda etapa do pipeline de dados esportivos. A partir do histórico bruto coletado (etapa de coleta), este notebook gera estatísticas consolidadas por time e analisa confrontos diretos, produzindo as bases que alimentam a análise de probabilidades e o modelo preditivo.

## Entrada
`loteca_historico_completo.csv` — histórico de concursos da Loteca (gerado pela etapa de coleta)

## Saída
- `analise_estatisticas_times.csv` — estatísticas consolidadas por time
- `analise_estatisticas_times_confrontos.csv` — confrontos diretos entre times

---

## Origem dos dados

Os dados foram coletados via web scraping do site oficial da Loteca (Caixa Econômica Federal). Ver módulo de coleta em `1_coleta/`.

## Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

## Análise e geração de estatísticas por time

In [ ]:
def analisar_historico_e_gerar_planilha(caminho_arquivo_original, caminho_arquivo_saida):
    """
    Analisa o histórico completo da Loteca e gera uma nova planilha com estatísticas detalhadas
    """
    # Carregar dados originais
    print("Carregando dados históricos...")
    df = pd.read_csv(caminho_arquivo_original, delimiter=';', encoding='utf-8')

    # Dicionário para armazenar estatísticas
    estatisticas = defaultdict(lambda: {
        'vitorias': 0,
        'empates': 0,
        'derrotas': 0,
        'gols_pro': 0,
        'gols_contra': 0,
        'jogos': 0
    })

    # Lista para armazenar os dados da nova planilha
    dados_nova_planilha = []

    print("Analisando confrontos...")

    # Processar cada jogo do histórico
    for idx, jogo in df.iterrows():
        time1 = jogo['time1']
        time2 = jogo['time2']
        gols_time1 = jogo['gols_time1']
        gols_time2 = jogo['gols_time2']
        resultado = jogo['resultado']

        # Atualizar estatísticas do time1
        estatisticas[time1]['jogos'] += 1
        estatisticas[time1]['gols_pro'] += gols_time1
        estatisticas[time1]['gols_contra'] += gols_time2

        # Atualizar estatísticas do time2
        estatisticas[time2]['jogos'] += 1
        estatisticas[time2]['gols_pro'] += gols_time2
        estatisticas[time2]['gols_contra'] += gols_time1

        # Determinar resultado para cada time
        if resultado == time1:
            # Time1 venceu
            estatisticas[time1]['vitorias'] += 1
            estatisticas[time2]['derrotas'] += 1
        elif resultado == time2:
            # Time2 venceu
            estatisticas[time2]['vitorias'] += 1
            estatisticas[time1]['derrotas'] += 1
        else:
            # Empate
            estatisticas[time1]['empates'] += 1
            estatisticas[time2]['empates'] += 1

    print("Gerando nova planilha...")

    # Obter todos os times únicos
    todos_times = list(set(df['time1'].unique()) | set(df['time2'].unique()))

    # Gerar dados para a nova planilha
    for time in todos_times:
        stats = estatisticas[time]

        dados_nova_planilha.append({
            'Time 1 - Nome': time,
            'Time 1 - Vitoria': stats['vitorias'],
            'Time 1 - Empate': stats['empates'],
            'Time 1 - Derrota': stats['derrotas'],
            'Time 2 - Nome': '',  # Esta coluna será preenchida posteriormente para análise de confrontos específicos
            'Time 2 - Vitoria': '',
            'Time 2 - Empate': '',
            'Time 2 - Derrota': ''
        })

    # Criar DataFrame da nova planilha
    df_nova_planilha = pd.DataFrame(dados_nova_planilha)

    # Ordenar por número de vitórias (decrescente)
    df_nova_planilha = df_nova_planilha.sort_values('Time 1 - Vitoria', ascending=False)

    # Salvar nova planilha
    df_nova_planilha.to_csv(caminho_arquivo_saida, index=False, encoding='utf-8')

    print(f"\nAnálise concluída!")
    print(f"Total de times analisados: {len(todos_times)}")
    print(f"Total de jogos processados: {len(df)}")
    print(f"Nova planilha salva como: {caminho_arquivo_saida}")

    return df_nova_planilha, estatisticas


## Análise de confrontos diretos

In [ ]:
def analisar_confrontos_especificos(caminho_arquivo_original, caminho_arquivo_saida):
    """
    Analisa confrontos específicos entre times (para uma análise mais detalhada)
    """
    # Carregar dados originais
    df = pd.read_csv(caminho_arquivo_original, delimiter=';', encoding='utf-8')

    # Lista para armazenar confrontos
    dados_confrontos = []

    print("\nAnalisando confrontos específicos entre times...")

    # Para cada time, analisar seus confrontos contra cada oponente
    todos_times = list(set(df['time1'].unique()) | set(df['time2'].unique()))

    for time1 in todos_times:
        # Encontrar todos os oponentes deste time
        oponentes_time1 = set()

        # Oponentes quando time1 jogou em casa
        oponentes_casa = df[df['time1'] == time1]['time2'].unique()
        # Oponentes quando time1 jogou fora
        oponentes_fora = df[df['time2'] == time1]['time1'].unique()

        oponentes_time1 = set(oponentes_casa) | set(oponentes_fora)

        for time2 in oponentes_time1:
            if time1 != time2:
                # Analisar confrontos diretos entre time1 e time2
                confrontos = df[
                    ((df['time1'] == time1) & (df['time2'] == time2)) |
                    ((df['time1'] == time2) & (df['time2'] == time1))
                ]

                vitorias_time1 = 0
                vitorias_time2 = 0
                empates = 0

                for _, jogo in confrontos.iterrows():
                    if jogo['resultado'] == time1:
                        vitorias_time1 += 1
                    elif jogo['resultado'] == time2:
                        vitorias_time2 += 1
                    else:
                        empates += 1

                # Adicionar à lista de confrontos
                dados_confrontos.append({
                    'Time 1 - Nome': time1,
                    'Time 1 - Vitoria': vitorias_time1,
                    'Time 1 - Empate': empates,
                    'Time 1 - Derrota': vitorias_time2,
                    'Time 2 - Nome': time2,
                    'Time 2 - Vitoria': vitorias_time2,
                    'Time 2 - Empate': empates,
                    'Time 2 - Derrota': vitorias_time1
                })

    # Criar DataFrame de confrontos
    df_confrontos = pd.DataFrame(dados_confrontos)

    # Salvar planilha de confrontos
    df_confrontos.to_csv(caminho_arquivo_saida.replace('.csv', '_confrontos.csv'), index=False, encoding='utf-8')

    print(f"Planilha de confrontos específicos salva!")

    return df_confrontos


## Estatísticas em destaque

In [ ]:
def mostrar_estatisticas_destaque(estatisticas):
    """
    Mostra estatísticas em destaque da análise
    """
    print("\n" + "="*60)
    print("ESTATÍSTICAS EM DESTAQUE")
    print("="*60)

    # Time com mais vitórias
    time_mais_vitorias = max(estatisticas.items(), key=lambda x: x[1]['vitorias'])
    print(f"Time com mais vitórias: {time_mais_vitorias[0]} ({time_mais_vitorias[1]['vitorias']} vitórias)")

    # Time com melhor aproveitamento
    times_aproveitamento = []
    for time, stats in estatisticas.items():
        if stats['jogos'] > 0:
            pontos = (stats['vitorias'] * 3) + stats['empates']
            aproveitamento = (pontos / (stats['jogos'] * 3)) * 100
            times_aproveitamento.append((time, aproveitamento, stats['jogos']))

    times_aproveitamento.sort(key=lambda x: x[1], reverse=True)

    print(f"\nTop 5 times com melhor aproveitamento:")
    for i, (time, apr, jogos) in enumerate(times_aproveitamento[:5]):
        print(f"{i+1}. {time}: {apr:.1f}% ({jogos} jogos)")

    # Estatísticas gerais
    total_jogos = sum(stats['jogos'] for stats in estatisticas.values()) // 2  # Divide por 2 porque cada jogo conta para 2 times
    total_gols = sum(stats['gols_pro'] for stats in estatisticas.values()) // 2

    print(f"\nEstatísticas gerais:")
    print(f"Total de jogos analisados: {total_jogos}")
    print(f"Total de gols marcados: {total_gols}")
    print(f"Média de gols por jogo: {total_gols/total_jogos:.2f}")


## Execução do pipeline

In [ ]:
# EXECUÇÃO PRINCIPAL
if __name__ == "__main__":
    # Configurações
    arquivo_original = 'loteca_historico_completo.csv'
    arquivo_saida = 'analise_estatisticas_times.csv'

    print("INICIANDO ANÁLISE DO HISTÓRICO DA LOTECA")
    print("="*50)

    try:
        # Executar análise principal
        df_nova_planilha, estatisticas = analisar_historico_e_gerar_planilha(
            arquivo_original,
            arquivo_saida
        )

        # Mostrar estatísticas em destaque
        mostrar_estatisticas_destaque(estatisticas)

        # Opcional: Analisar confrontos específicos
        analisar_confrontos_especificos(arquivo_original, arquivo_saida)

        print("\n" + "="*50)
        print("ANÁLISE CONCLUÍDA COM SUCESSO!")
        print("="*50)
        print("\nArquivos gerados:")
        print(f"1. {arquivo_saida} - Estatísticas gerais de cada time")
        print(f"2. {arquivo_saida.replace('.csv', '_confrontos.csv')} - Confrontos específicos entre times")

        # Mostrar preview da nova planilha
        print(f"\nPreview da nova planilha (primeiras 5 linhas):")
        print(df_nova_planilha.head())

    except FileNotFoundError:
        print(f"ERRO: Arquivo '{arquivo_original}' não encontrado.")
        print("Verifique se o arquivo está no diretório correto.")
    except Exception as e:
        print(f"ERRO durante a análise: {e}")

INICIANDO ANÁLISE DO HISTÓRICO DA LOTECA
Carregando dados históricos...
Analisando confrontos...
Gerando nova planilha...

Análise concluída!
Total de times analisados: 1080
Total de jogos processados: 17000
Nova planilha salva como: analise_estatisticas_times.csv

ESTATÍSTICAS EM DESTAQUE
Time com mais vitórias: FLAMENGO/RJ (415 vitórias)

Top 5 times com melhor aproveitamento:
1. INTER/RS: 100.0% (1 jogos)
2. CELTA: 100.0% (1 jogos)
3. BARBARENSE/SP: 100.0% (1 jogos)
4. GREMIO MARINGA/PR: 100.0% (1 jogos)
5. ALEGRENSE/ES: 100.0% (1 jogos)

Estatísticas gerais:
Total de jogos analisados: 17000
Total de gols marcados: 22146
Média de gols por jogo: 1.30

Analisando confrontos específicos entre times...
Planilha de confrontos específicos salva!

ANÁLISE CONCLUÍDA COM SUCESSO!

Arquivos gerados:
1. analise_estatisticas_times.csv - Estatísticas gerais de cada time
2. analise_estatisticas_times_confrontos.csv - Confrontos específicos entre times

Preview da nova planilha (primeiras 5 linhas